# Lecture 12 — Dashboard Design & Polish


> **Dataset:** Airbnb London

> **Big Book of Dashboards:** Ch.33 Red/Green + Ch.34 Pies/Donuts 


---
## Opening: Model Answer Review


---
## Design Principles - Part I: Avoiding Design Traps


### Trap 1: Red and Green (BBD Ch.33)

> **💡 ~8% of men have a colour vision deficiency — in a room of nine men, one cannot tell your red from your green**

> **💡 The fix: blue (#2E75B6) + orange (#E07B39) instead of red/green — distinguishable by almost all CVD viewers**

**Advanced fix:** provide a toggle to swap the palette (BBD Fig. 33.6)

### Trap 2: Pies and Donuts (BBD Ch.34)

> **💡 You WILL be asked for pie charts — accommodate and add a companion bar chart: the pie satisfies the request, the bar does the analysis**

**The donut KPI exception** — a donut is acceptable as a KPI indicator only when:
- The upper bound is exactly 100% (e.g. % of capacity used, % of seats filled)
- You are NOT comparing donut to donut (12 donuts side by side is a trap)
- A progress bar would work equally well (and is usually cleaner)


---
## Design Principles - Part II: The 5-Second Test & Polish


### The 5-second test

> **💡 Show your dashboard for 5 seconds, hide the screen, ask: What is it about? What is the most important number? What needs attention? If they cannot answer, the design has failed before any data has been read**

**How to pass:**
- Big KPI cards at the very top — readable at a glance
- Insight titles on every chart — the finding, not the topic
- One dominant colour per page — the eye knows where to look
- Minimal clutter — white background, light gridlines

### Streamlit polish techniques

```python
# Custom CSS — applied once in app.py (it runs on every page switch)
st.markdown("""
<style>
.block-container { padding-top: 1.5rem; padding-bottom: 0; }
[data-testid='metric-container'] {
    background: #F8F9FA; border: 1px solid #E9ECEF;
    padding: 1rem; border-radius: 8px;
}
footer { visibility: hidden; }
</style>
""", unsafe_allow_html=True)
```

```python
# st.expander — hide detail gracefully (BBD: progressive disclosure)
with st.expander('Show raw data'):
    st.dataframe(df)
```

```python
# Data freshness footer — always tell the audience when data was last updated
import datetime
st.caption(f'Data source: Inside Airbnb | Last updated: {datetime.date.today()}')
```


---
## Design Principles - Part III: Persisting Filters Between Pages


> **💡 A widget with `key=` mirrors its value into `st.session_state[key]`**

> **💡 The gotcha: Streamlit deletes a widget's key when a run finishes without rendering that widget — two pages = two scripts, so naive per-page filters reset on every switch**

> **💡 The fix: re-assign the key at the top of every run — `st.session_state[key] = st.session_state[key]` — this marks it app-owned so it survives page switches**

```python
# The keep-alive pattern — run BEFORE the widget is created
for key in ['flt_rooms', 'flt_hoods', 'flt_price']:
    if key in st.session_state:
        st.session_state[key] = st.session_state[key]  # keep alive across pages
```

> **💡 Architecture consequence: put ONE `sidebar_filters()` function in `utils.py` and call it on every page — same widgets, same keys, same filtered dataframe**

```python
# Every page starts the same way:
df, p95 = load_data()
filtered = sidebar_filters(df, p95)
```

**Widget key rule:** once a key is initialised in session_state, create the widget with `key=` only — do NOT also pass `default=`/`value=`, or Streamlit warns that the value is being set twice


---
## Let's Code An Example Dashboard 💻 

> **HOW TO RUN:**

> Build file by file: utils.py → app.py → pages/01 → pages/02
> 
> Show each page in the browser as you build it
> 
> Demo the persistence explicitly: set filters on page 1, switch to page 2, switch back


In [ ]:
import pandas as pd


df = pd.read_csv('../data/airbnb_london.csv')

df.head()

---
## Transition to Exercise

> 1. You receive the complete 2-page app we just built as starter code
> 2. Add a third page — "Where is guest demand strongest?" — the next level of the squiggle: market summary → neighbourhood story → demand
> 3. Do not modify pages 1 and 2. Push the whole folder to week12/ in your classwork repo

**BBD + Streamlit checklist for the new page:**
- Page title is a question (not 'Demand' or 'Details')
- Registered in `app.py` navigation with an icon
- `sidebar_filters()` called at the top — shared filters persist onto your page
- Your own widget persisted too (keep-alive + guard for filtered-out values)
- Colour type named in a comment (highlight: blue vs grey)
- No red-green as only differentiator, no pies, no packed bubbles
- KPI row passes the 5-second test
- Insight title on the chart, white background, Arial
